In [1]:
"""
Whole Pipeline Testing 

Author: Tan Yi Feng
Student ID: 23WMR14766
"""

import pandas as pd
import numpy as np
import joblib
import yaml
import sys
import pickle
import logging
import time
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, 
    precision_score,
    recall_score,
    f1_score,
    classification_report, 
    confusion_matrix
)

from pathlib import Path
import joblib
import pickle
import pandas as pd

SCRIPT_DIR = Path.cwd()

DATASET_PATH = SCRIPT_DIR / 'data' / 'processed' / 'full_inference_cleaned_casing_preserved.csv'
CONFIG_PATH = SCRIPT_DIR / 'config' / 'config.yaml'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'
SRC_PATH = SCRIPT_DIR / 'src'

sys.path.append(str(SRC_PATH))

from preprocessing.feature_extractor import FeatureExtractor

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
logger = logging.getLogger(__name__)

ATTACK_LABELS = {
    0: "Benign (Normal)",
    1: "ARP Spoofing", 2: "MQTT Connect Flood", 3: "MQTT Publish Flood",
    4: "MQTT Malformed", 5: "Reconnaissance", 6: "Recon (VulnScan)",
    7: "ICMP Flood", 8: "SYN Flood", 9: "TCP Flood", 10: "UDP Flood"
}

def load_config(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def run_nodal_test(dataset_path):
    print(" STARTING DUAL-LAYER IDS TEST...")
    
    # Start total timer
    total_start_time = time.time()
    
 
    try:
        config = load_config(CONFIG_PATH)
    except Exception as e:
        print(f"❌ Config Load Error: {e}")
        return

   
    print("\n[STEP 1] Loading Models...")
    try:
        ensemble = joblib.load(MODELS_DIR / 'ensemble.joblib')
        scaler = joblib.load(MODELS_DIR / 'scaler.joblib')
        
        rf_pipeline = joblib.load(MODELS_DIR / 'best_rf_pipeline2.pkl')
        with open(MODELS_DIR / 'selected_features2.pkl', 'rb') as f:
            selected_features = list(pickle.load(f))
            
        print(f"✅ Models Loaded Successfully.")
        print(f"   - L2 Features: {len(selected_features)}")
        
    except FileNotFoundError as e:
        print(f"❌ Error: {e}")
        return

    
    print(f"\n[STEP 2] Loading Data: {dataset_path}")
    try:
        df = pd.read_csv(dataset_path) 
        print(f"   - Loaded {len(df):,} rows")
        total_packets = len(df)
    except Exception as e:
        print(f"❌ Data Load Error: {e}")
        return

    print("\n[STEP 3] Running Layer 1 (Anomaly Detection)...")
    layer1_start = time.time()
    
    extractor = FeatureExtractor()
    features_df = extractor.extract_from_dataframe(df)
    features_norm, _ = extractor.normalize_features(features_df, scaler)
    X_L1 = extractor.get_feature_vector(features_norm)
    
    predictions = ensemble.predict_with_details(X_L1)
    
    threshold = config.get('ensemble', {}).get('threshold_low', 0.1)
    layer1_flags = predictions['ensemble_score'] > threshold
    suspicious_indices = np.where(layer1_flags)[0]
    
    layer1_time = time.time() - layer1_start
    
    print(f"   ✅ Layer 1 Scan Complete.")
    print(f"   - Normal:     {len(df) - len(suspicious_indices):,}")
    print(f"   - Suspicious: {len(suspicious_indices):,} (Passed to L2)")
    
    if 'label' in df.columns:
        y_true_binary = (df['label'] != 0).astype(int)  # 0 = Benign, 1 = Attack
        y_pred_binary = layer1_flags.astype(int)         # 0 = Normal, 1 = Suspicious
        
        binary_acc = accuracy_score(y_true_binary, y_pred_binary)
        binary_precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        binary_recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        binary_f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        
        print("\n" + "="*40)
        print("   LAYER 1: BINARY CLASSIFICATION")
        print("="*40)
        print(f"Accuracy   : {binary_acc:.4f}")
        print(f"Precision  : {binary_precision:.4f}")
        print(f"Recall     : {binary_recall:.4f}")
        print(f"F1-Score   : {binary_f1:.4f}")


    print("\n[STEP 4] Running Layer 2 (Attack Classification)...")
    layer2_start = time.time()
    
   
    final_attack_ids = np.zeros(len(df), dtype=int)
    
    if len(suspicious_indices) > 0:
        try:
           
            X_L2_subset = df.loc[suspicious_indices, selected_features].copy()
            
         
            X_L2_subset.replace([np.inf, -np.inf], np.nan, inplace=True)
            imputer = SimpleImputer(strategy='median')
            X_L2_clean = imputer.fit_transform(X_L2_subset)
            
           
            probs = rf_pipeline.predict_proba(X_L2_clean)
            max_probs = np.max(probs, axis=1)
            raw_preds = np.argmax(probs, axis=1)
            
          
            mapped_preds = raw_preds + 1
            l2_decisions = np.where(max_probs >= 0.9, mapped_preds, 99)
            
            final_attack_ids[suspicious_indices] = l2_decisions
            
        except KeyError as e:
            print(f"❌ COLUMN ERROR: {e}")
            return
    
    layer2_time = time.time() - layer2_start
    
    total_time = time.time() - total_start_time
    avg_latency = total_time / total_packets
  
    if 'label' in df.columns:
        y_true = df['label'].astype(str).str.split('.').str[0].astype(int)
        y_pred = final_attack_ids

        ambiguous_count = np.sum(y_pred == 99)
        certain_count = len(y_pred) - ambiguous_count

        print("\n" + "="*40)
        print("      LAYER 2: MULTI-CLASS RESULTS")
        print("="*40)
        print(f"Confidence Threshold  : 0.9")
        print(f"Ambiguous (Label 99)  : {ambiguous_count} ({(ambiguous_count/len(y_pred))*100:.2f}%)")
        print(f"High Confidence       : {certain_count} ({(certain_count/len(y_pred))*100:.2f}%)")

        mask = y_pred != 99
        
        if np.any(mask):
            y_true_certain = y_true[mask]
            y_pred_certain = y_pred[mask]
            
            acc = accuracy_score(y_true_certain, y_pred_certain)
            print(f"Accuracy (Certain)    : {acc:.4f}")
            
            print("\nDetailed Classification Report (High Confidence Only):")
            
           
            unique_pred_labels = np.unique(y_pred_certain)
            
            unique_pred_labels = unique_pred_labels[unique_pred_labels != 0]
            
            target_names = [ATTACK_LABELS.get(l, f"Class {l}") for l in unique_pred_labels]
            
            print(classification_report(
                y_true_certain, 
                y_pred_certain, 
                labels=unique_pred_labels,
                target_names=target_names,
                digits=4,
                zero_division=0
            ))
        else:
            print("\n⚠️ No confident predictions were made.")

    else:
        print("⚠️ No labels found. Skipping evaluation.")

    
    print("\n" + "="*40)
    print("        PERFORMANCE METRICS")
    print("="*40)
    print(f"Total Packets         : {total_packets:,}")
    print(f"Layer 1 Time          : {layer1_time:.4f} s")
    print(f"Layer 2 Time          : {layer2_time:.4f} s")
    print(f"Total Detection Time  : {total_time:.4f} s")
    print(f"Avg Latency/Packet    : {avg_latency:.8f} s ({avg_latency*1000:.4f} ms)")
    print(f"Throughput            : {total_packets/total_time:.2f} packets/s")

    print("\n✅ Test Finished.")

run_nodal_test(DATASET_PATH)

🚀 STARTING DUAL-LAYER IDS TEST...

[STEP 1] Loading Models...
✅ Models Loaded Successfully.
   - L2 Features: 25

[STEP 2] Loading Data: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv


Extracting features from 6,987,394 flows...
Input columns: 41


   - Loaded 6,987,394 rows

[STEP 3] Running Layer 1 (Anomaly Detection)...


[OK] Extracted 67 columns
     - 63 ML features
     - Metadata: timestamp, protocol, label, scenario
     - NO synthetic IPs/ports (removed)
C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
[OK] Features normalized (detection mode - used existing scaler)


   ✅ Layer 1 Scan Complete.
   - Normal:     17,813
   - Suspicious: 6,969,581 (Passed to L2)

   LAYER 1: BINARY CLASSIFICATION
Accuracy   : 0.9994
Precision  : 0.9996
Recall     : 0.9998
F1-Score   : 0.9997

[STEP 4] Running Layer 2 (Attack Classification)...


C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



      LAYER 2: MULTI-CLASS RESULTS
Confidence Threshold  : 0.9
Ambiguous (Label 99)  : 64267 (0.92%)
High Confidence       : 6923127 (99.08%)
Accuracy (Certain)    : 0.9993

Detailed Classification Report (High Confidence Only):
                    precision    recall  f1-score   support

      ARP Spoofing     0.8255    0.8517    0.8384      8735
MQTT Connect Flood     0.9999    0.9998    0.9998    181083
MQTT Publish Flood     0.9961    0.9987    0.9974     70483
    MQTT Malformed     0.8125    0.9594    0.8799      3668
    Reconnaissance     0.9972    0.9832    0.9902     95343
  Recon (VulnScan)     0.7161    0.9852    0.8294       676
        ICMP Flood     0.9999    0.9999    0.9999   1949442
         SYN Flood     1.0000    0.9999    1.0000   1232361
         TCP Flood     1.0000    0.9997    0.9999   1171605
         UDP Flood     1.0000    0.9997    0.9998   2193114

         micro avg     0.9995    0.9993    0.9994   6906510
         macro avg     0.9347    0.9777    0.953